# Phos Qwen3-VL-4B 微调笔记本（Google Colab 1,000 积分版）

**适配底模**：`Qwen/Qwen3-VL-4B-Instruct` → 转换回 Mac 本机 `mlx-community/Qwen3-VL-4B-Instruct-4bit` (`mlx-vlm 0.7.8`)

### 推荐执行节奏（节省积分不翻车）
1. **Phase 0 冒烟验证（选 NVIDIA L4，~15 分钟，约 1–2 CU）**：用 `smoke200` 跑 1 epoch → 下载 `smoke_peft_adapter.zip` 回 Mac 跑 `convert_peft_to_phos_mlx.py --verify-mlx`，确认全链路 100% 跑通。
2. **Phase 2 主训练（切 NVIDIA A100 80GB，~380 CU 预算）**：全 36 层 `language_model` 7 大投影（`q/k/v/o/gate/up/down`），`rank=32, alpha=64`，自动向 Google Drive 持久化断点续训。
3. **Phase 3 视觉闭环 DPO（A100 80GB，~260 CU 预算）**：用 Phos 本机渲染打分构造 `(chosen, rejected)` 后再跑 2 epoch DPO。

In [ ]:
# === Cell 1: 检查 GPU 型号与挂载 Google Drive ===
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
# 请把 Mac 上生成的 Phos-Colab-Ready-*.zip 上传到 Google Drive 的 MyDrive/PhosTrain/ 目录
DRIVE_ROOT = Path('/content/drive/MyDrive/PhosTrain')
BUNDLE_ZIP = DRIVE_ROOT / 'Phos-Colab-Ready-20261125.zip'
WORK_DIR = Path('/content/phos_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Drive 目录存在:', DRIVE_ROOT.exists(), '| 上传包存在:', BUNDLE_ZIP.exists())

In [ ]:
# === Cell 2: 安装训练依赖（与 Qwen3-VL 兼容）并解压数据集到本地高速 SSD ===
!pip install -qU "transformers>=4.57.0" "peft>=0.14.0" "trl>=0.15.0" "accelerate>=1.3.0" "datasets>=3.2.0" "qwen-vl-utils>=0.0.10" "safetensors>=0.4.5" "pillow"

import zipfile
DATA_LOCAL = WORK_DIR / 'dataset'
if not (DATA_LOCAL / 'dataset-summary.json').exists():
    assert BUNDLE_ZIP.exists(), f'请先上传数据集包到 {BUNDLE_ZIP}'
    with zipfile.ZipFile(BUNDLE_ZIP, 'r') as zf:
        zf.extractall(DATA_LOCAL)
print((DATA_LOCAL / 'dataset-summary.json').read_text(encoding='utf-8'))

In [ ]:
# === Cell 3: 训练前协议与图片完整性快速自检（零积分浪费） ===
import json, sys
sys.path.insert(0, str(DATA_LOCAL / 'scripts'))
from photo_ai_protocol import VERSION, FINGERPRINT, serialize_target

for split_dir in ('smoke200', 'full'):
    for fn in ('train.jsonl', 'validation.jsonl'):
        fp = DATA_LOCAL / split_dir / fn
        if not fp.exists(): continue
        rows = [json.loads(x) for x in fp.read_text(encoding='utf-8').splitlines() if x.strip()]
        missing_imgs = sum(1 for r in rows for img in r['images'] if not (DATA_LOCAL / img).is_file())
        non_canon = sum(1 for r in rows if r['messages'][-1]['content'] != serialize_target(json.loads(r['messages'][-1]['content'])))
        print(f'{split_dir}/{fn}: {len(rows)} rows | missing_imgs={missing_imgs} | non_canon={non_canon}')
        assert missing_imgs == 0 and non_canon == 0, '协议或图像校验失败！'
print(f'✓ 全部样本通过 {VERSION} ({FINGERPRINT[:12]}...) 契约自检！')

## Phase 0：冒烟验证（建议先用 L4 跑，耗时约 12–15 分钟，消耗 < 2 CU）
跑完后会把 `smoke_peft_adapter.zip` 存到 Google Drive，先下载回 Mac 用 `convert_peft_to_phos_mlx.py --verify-mlx` 实机验证加载通过，再切 A100 跑全量！

In [ ]:
# === Cell 4: Phase 0 冒烟训练（smoke200, 1 epoch, r=16） ===
SMOKE_OUT = DRIVE_ROOT / 'runs/phase0_smoke_r16'
!python {DATA_LOCAL}/scripts/colab_train_phos_qwen3_vl.py \
  --stage sft \
  --model Qwen/Qwen3-VL-4B-Instruct \
  --data-dir {DATA_LOCAL}/smoke200 \
  --image-root {DATA_LOCAL} \
  --out-dir {SMOKE_OUT} \
  --rank 16 --epochs 1 --batch-size 1 --grad-accum 8 --lr 1e-4 \
  --eval-steps 25 --save-steps 25

# 打包冒烟权重方便下载回 Mac 测试
!cd {SMOKE_OUT} && zip -rq {DRIVE_ROOT}/smoke_peft_adapter.zip final_peft_adapter
print('✓ 冒烟权重已打包：', DRIVE_ROOT / 'smoke_peft_adapter.zip')

## Phase 2：全 36 层 SFT 主训练（切到 **A100 80GB** 运行）
- 支持自动断点续训：若 Colab 12 小时断连，重新运行 Cell 1–2 后，把 `RESUME_CKPT` 设为 `"true"`（或具体 `checkpoint-XXXX` 路径）即可无缝接着训。

In [ ]:
# === Cell 5: Phase 2 全 36 层主训练（A100 80GB, r=32, alpha=64, 3 epochs） ===
SFT_OUT = DRIVE_ROOT / 'runs/phase2_sft_v2_r32'
RESUME_CKPT = ''  # 断连续训时改为 'true' 或具体 checkpoint 路径
resume_arg = f'--resume {RESUME_CKPT}' if RESUME_CKPT else ''

!python {DATA_LOCAL}/scripts/colab_train_phos_qwen3_vl.py \
  --stage sft \
  --model Qwen/Qwen3-VL-4B-Instruct \
  --data-dir {DATA_LOCAL}/full \
  --image-root {DATA_LOCAL} \
  --out-dir {SFT_OUT} \
  --rank 32 --epochs 3 --batch-size 2 --grad-accum 8 --lr 1.2e-4 \
  --eval-steps 150 --save-steps 150 {resume_arg}

!cd {SFT_OUT} && zip -rq {DRIVE_ROOT}/phase2_sft_peft_adapter.zip final_peft_adapter
print('✓ Phase 2 SFT 权重已保存：', DRIVE_ROOT / 'phase2_sft_peft_adapter.zip')

## Phase 3：视觉闭环 DPO 偏好对齐（可选，准备好 `dpo_train.jsonl` / `dpo_val.jsonl` 后运行）

In [ ]:
# === Cell 6: Phase 3 DPO 偏好对齐（A100 80GB） ===
DPO_DATA = DRIVE_ROOT / 'dpo_data'
DPO_OUT = DRIVE_ROOT / 'runs/phase3_dpo_v2_r32'

if (DPO_DATA / 'dpo_train.jsonl').exists():
    !python {DATA_LOCAL}/scripts/colab_train_phos_qwen3_vl.py \
      --stage dpo \
      --model Qwen/Qwen3-VL-4B-Instruct \
      --init-adapter {DRIVE_ROOT}/runs/phase2_sft_v2_r32/final_peft_adapter \
      --data-dir {DPO_DATA} \
      --image-root {DATA_LOCAL} \
      --out-dir {DPO_OUT} \
      --epochs 2 --grad-accum 16 --lr 1.5e-5 --dpo-beta 0.1 \
      --eval-steps 100 --save-steps 100
    !cd {DPO_OUT} && zip -rq {DRIVE_ROOT}/phase3_dpo_peft_adapter.zip final_dpo_peft_adapter
    print('✓ Phase 3 DPO 权重已保存：', DRIVE_ROOT / 'phase3_dpo_peft_adapter.zip')
else:
    print('尚未放置 dpo_train.jsonl，跳过 Phase 3。')

## 回传 Mac 一键转换与部署命令
将 Google Drive 上的 `phase2_sft_peft_adapter.zip`（或 `phase3_dpo_peft_adapter.zip`）下载并解压到 Mac 后，在终端运行：
```bash
cd "/Users/yangxiaoying/Desktop/Phos资料/源码-RawForge-v2.0.0"
"$HOME/Library/Application Support/Phos/MLX/.venv/bin/python" Scripts/convert_peft_to_phos_mlx.py \
  --peft-dir ~/Downloads/final_peft_adapter \
  --out-dir ~/Downloads/phos-mlx-adapter-v2 \
  --name "Phos Qwen3-VL-4B Full-Layer v2" \
  --verify-mlx
```
验证通过后覆盖到 `~/Library/Application Support/Phos/MLX/adapter/` 即可在 Phos 内启用。